# 01 — Environment test, data sanity check, real-model smoke test (Gate G1 → G2)

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = Files.

This notebook does only the day-one steps. It **stops** after the smoke test: copy the printed
outputs of cells 2, 4, 5, 6 and 8 back into the chat / `research_decisions.md` before running any
experiment. Nothing in this notebook is a result.

## 1. Get the repository

In [ ]:
# Option A (after you push the repo to GitHub):
REPO_URL = "https://github.com/rovermask/AITR_Lab02_Project.git"   # <- change if different
# Option B: upload the zip as a Kaggle Dataset and set ZIP_PATH instead.
ZIP_PATH = None   # e.g. "/kaggle/input/aitr-lab02/AITR_Lab02_Project.zip"

import os, subprocess
os.chdir("/kaggle/working")
if ZIP_PATH:
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", "/kaggle/working"], check=True)
elif not os.path.exists("AITR_Lab02_Project"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("/kaggle/working/AITR_Lab02_Project")
print(os.getcwd()); print(sorted(os.listdir(".")))

## 2. GPU / system diagnostics

In [ ]:
!nvidia-smi
!python --version
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
      "| devices:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 3. Install dependencies
`vllm` is deliberately **not pinned** yet: its compatibility with Kaggle's current CUDA/torch image is
**[TEST REQUIRED]**. Whatever version installs and passes cell 8 gets pinned in `requirements-gpu.txt`.
If vllm fails on T4, fall back to `--backend hf` (slow) and log it.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets vllm
import importlib.metadata as md
for p in ["vllm","torch","transformers","datasets","numpy","pandas","scipy"]:
    try: print(p, md.version(p))
    except Exception as e: print(p, "NOT INSTALLED", e)

## 4. Raw MedQA test split + integrity check against the hash recorded on 2026-10-20

In [ ]:
import hashlib, os, subprocess
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/medqa_test.jsonl"):
    subprocess.run(["git","clone","--depth","1","https://github.com/gersteinlab/MedAgents.git","/tmp/MedAgents"], check=True)
    subprocess.run(["cp","/tmp/MedAgents/datasets/MedQA/test.jsonl","data/raw/medqa_test.jsonl"], check=True)
h = hashlib.sha256(open("data/raw/medqa_test.jsonl","rb").read()).hexdigest()
EXPECTED = "cfa75cbcc53c29b023752b0a395ff49d8202121041392735ae44771a5ec734a3"
print(h); print("HASH MATCH" if h == EXPECTED else "HASH MISMATCH — stop and log this")
!python scripts/prepare_data.py --src data/raw/medqa_test.jsonl --out data/processed

## 5. Hugging Face datasets: MedMCQA-dev oncology subset, MedQA 4-option test, pilot items

In [ ]:
!python scripts/prepare_extra_data.py --out data/processed

## 6. Unit tests (CPU, mock backend — checks the code, not the model)

In [ ]:
!python -m pytest -q tests

## 7. Load the real model
Primary: `Qwen/Qwen2.5-7B-Instruct`, fp16, tensor-parallel over both T4s (config default).
If this OOMs: set `tensor_parallel_size: 1`, `quantization: awq`, model `Qwen/Qwen2.5-7B-Instruct-AWQ`
(log the change). Record load time.

## 8. Smoke test on 5 MedQA **train** pilot items (never reported) — CoT, then one board run

In [ ]:
import time, json, subprocess
t=time.time()
subprocess.run(["python","scripts/run_experiment.py","--dataset","pilot_train","--method","cot",
                "--seed","0","--backend","vllm","--limit","5","--out","results/pilot"], check=True)
print(f"CoT x5 incl. model load: {time.time()-t:.0f}s")
for r in map(json.loads, open("results/pilot/pilot_train__cot__Qwen2.5-7B-Instruct__seed0.jsonl")):
    print("="*80); print("gold:", r["gold"], "| parsed:", r["final"], "| tokens:", r["prompt_tokens"], r["completion_tokens"])
    print(r["samples"][0]["text"][-600:])

In [ ]:
# One board run on 3 items: measures real per-item cost of the 9-call pipeline (needed for the compute plan).
import time, subprocess, json
t=time.time()
subprocess.run(["python","scripts/run_experiment.py","--dataset","pilot_train","--method","board_roles",
                "--seed","0","--backend","vllm","--limit","3","--chunk","3","--out","results/pilot"], check=True)
print(f"board_roles x3 incl. model load: {time.time()-t:.0f}s")
for r in map(json.loads, open("results/pilot/pilot_train__board_roles__Qwen2.5-7B-Instruct__seed0.jsonl")):
    print(r["item_id"], "gold", r["gold"], "| R1", [m["answer"] for m in r["rounds"][0].values()],
          "| R2", [m["answer"] for m in r["rounds"][1].values()], "| chair", r["final"],
          "| api tokens", r["prompt_tokens"]+r["completion_tokens"])

## STOP HERE
Paste back: cell 2 (GPU), cell 3 (versions), cell 4 (hash + counts), cell 5 (MedMCQA counts),
cell 6 (test summary), cell 8 (both timings, parsed vs gold, any parse failures).
Next step is Gate G2: baseline validation (`baseline_validation.md`), **not** the main experiment.